# Colab Template: Interactive Dashboard with Streamlit + ngrok

**Goal:** Run a Streamlit dashboard from Google Colab and expose it via a public URL using **ngrok**.

Works well for classroom demos and student assignments.

## Notes
- You need an ngrok **authtoken** (free). Get it from your ngrok dashboard.
- This notebook writes a small Streamlit app (`app.py`) and runs it in the background.
- You will receive a clickable public URL.

---
## Dataset
This template generates a small synthetic sales dataset (date, region, category, channel, units, price, revenue). You can replace it with your real dataset later.


## 1) Install dependencies

In [1]:
!pip -q install streamlit pyngrok plotly pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 91.3 MB/s eta 0:00:00


## 2) Configure ngrok

### Option A (recommended): set your token via a Colab "secret" or directly
- Replace `YOUR_NGROK_AUTHTOKEN_HERE` with your token.
- Keep your token private.


In [2]:
from google.colab import userdata
NGROK_AUTH_TOKEN= userdata.get('ngrok')

In [3]:
from pyngrok import ngrok

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
else:
    print("⚠️ Please set NGROK_AUTH_TOKEN to your token before continuing.")

## 3) Write the Streamlit dashboard app
This creates an `app.py` file in the Colab runtime.

In [4]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px

st.set_page_config(page_title="Sales Dashboard", layout="wide")

st.title("📊 Sales Dashboard (Streamlit + Colab)")
st.caption("Interactive dashboard demo running inside Google Colab using a tunnel.")

# ------------------------
# Data generation
# ------------------------
@st.cache_data
def make_sales_data(n: int = 3000, seed: int = 7) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2025-01-01", "2025-12-31", freq="D")
    regions = ["North", "South", "East", "West"]
    categories = ["A", "B", "C"]
    channels = ["Online", "Retail"]

    df = pd.DataFrame({
        "date": rng.choice(dates, size=n),
        "region": rng.choice(regions, size=n),
        "category": rng.choice(categories, size=n, p=[0.45, 0.35, 0.20]),
        "channel": rng.choice(channels, size=n, p=[0.6, 0.4]),
        "units": rng.integers(1, 8, size=n),
    })
    base_price = df["category"].map({"A": 18, "B": 28, "C": 40}).astype(float)
    noise = rng.normal(0, 3, size=n)
    df["price"] = (base_price + noise).clip(5, None).round(2)
    df["revenue"] = (df["units"] * df["price"]).round(2)
    df["date"] = pd.to_datetime(df["date"])
    df["month"] = df["date"].dt.to_period("M").astype(str)
    return df

df = make_sales_data()

# ------------------------
# Sidebar filters
# ------------------------
st.sidebar.header("Filters")
regions = st.sidebar.multiselect("Region", sorted(df["region"].unique()), default=sorted(df["region"].unique()))
cats = st.sidebar.multiselect("Category", sorted(df["category"].unique()), default=sorted(df["category"].unique()))
chans = st.sidebar.multiselect("Channel", sorted(df["channel"].unique()), default=sorted(df["channel"].unique()))

months = sorted(df["month"].unique())
m_lo, m_hi = st.sidebar.select_slider("Month range", options=months, value=(months[0], months[-1]))

remove_top = st.sidebar.checkbox("Remove top 1% revenue (outlier filter)", value=False)

# Apply filters
dff = df[(df["region"].isin(regions)) & (df["category"].isin(cats)) & (df["channel"].isin(chans))]
dff = dff[(dff["month"] >= m_lo) & (dff["month"] <= m_hi)]

if remove_top and len(dff) > 0:
    cap = dff["revenue"].quantile(0.99)
    dff = dff[dff["revenue"] <= cap]


# ------------------------
# KPI row
# ------------------------
c1, c2, c3, c4 = st.columns(4)
c1.metric("Total Revenue", f"{dff['revenue'].sum():,.2f}")
c2.metric("Total Units", f"{int(dff['units'].sum()):,}")
c3.metric("Avg Price", f"{dff['price'].mean():,.2f}" if len(dff) else "0.00")
c4.metric("Rows", f"{len(dff):,}")

# --------------
# Additional Charts
#######

# ------------------------
# 6) Student Tasks (quick assignment)
# ------------------------
st.markdown("### 6) Student Tasks")

# 1) Histogram of price with a slider for bin count
bins = st.slider("Number of bins for price histogram", min_value=5, max_value=100, value=30, step=1)
fig_hist = px.histogram(
    dff, x="price", nbins=bins,
    title=f"Price distribution ({bins} bins)",
    template="plotly_white"
)
st.plotly_chart(fig_hist, use_container_width=True)

# 2) Line chart of units over time
# If there are multiple rows per day, aggregate first
units_over_time = (
    dff.groupby("month", as_index=False)["units"]
       .sum()
       .sort_values("month")
)
fig_units = px.line(
    units_over_time, x="month", y="units",
    markers=True, title="Units Over Time",
    template="plotly_white"
)
st.plotly_chart(fig_units, use_container_width=True)

# 3) Download button to export the *filtered* dataframe
csv_bytes = dff.to_csv(index=False).encode("utf-8")
st.download_button(
    label="⬇️ Download filtered data as CSV",
    data=csv_bytes,
    file_name="filtered_sales.csv",
    mime="text/csv",
)


# ------------------------
# Existing Charts
# ------------------------
left, right = st.columns([1.35, 1])

with left:
    by_date = dff.groupby("date", as_index=False)["revenue"].sum().sort_values("date")
    fig_trend = px.line(by_date, x="date", y="revenue", title="Revenue Over Time")
    st.plotly_chart(fig_trend, use_container_width=True)

with right:
    by_cat = dff.groupby("category", as_index=False)["revenue"].sum().sort_values("revenue", ascending=False)
    fig_bar = px.bar(by_cat, x="category", y="revenue", title="Revenue by Category")
    st.plotly_chart(fig_bar, use_container_width=True)

st.subheader("Preview (Top 20 by revenue)")
st.dataframe(dff.sort_values("revenue", ascending=False).head(20), use_container_width=True)


Writing app.py


## 4) Run Streamlit and expose it with ngrok

This cell:
1. Stops any existing Streamlit process
2. Starts Streamlit in the background
3. Creates a public ngrok URL you can click


In [5]:
import os
import time
from pyngrok import ngrok

# Kill any previous streamlit servers
!pkill -f streamlit || true

# Start Streamlit
get_ipython().system_raw("streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &")

# Give it a moment to start
time.sleep(2)

# Close any existing tunnels on 8501
for t in ngrok.get_tunnels():
    if "8501" in t.public_url or "8501" in t.config.get("addr", ""):
        try:
            ngrok.disconnect(t.public_url)
        except Exception:
            pass

# Create a new tunnel
public_url = ngrok.connect(addr=8501, proto="http")
print("✅ Streamlit is running!")
print("🌐 Public URL:", public_url)
print("\nIf you see errors, open streamlit.log for details.")

^C
✅ Streamlit is running!
🌐 Public URL: NgrokTunnel: "https://squirmiest-israel-freckly.ngrok-free.dev" -> "http://localhost:8501"

If you see errors, open streamlit.log for details.


## 5) Troubleshooting

**A) The URL opens but shows an error**
- Check the Streamlit logs:
```python
!tail -n 50 streamlit.log
```

**B) ngrok says authentication failed**
- Ensure you set the correct `NGROK_AUTH_TOKEN`.

**C) Port already in use**
- Rerun the run cell (it kills old processes first).

**D) Students without ngrok tokens**
- Use **localtunnel** as an alternative, or run locally.


## 6) Student Tasks (quick assignment)
1. Add a histogram of `price` with a slider for bin count.
2. Add a line chart of `units` over time.
3. Add a download button to export the filtered dataframe.
